# pinn_ndof_chain_sim_tf2_free_free_icv.ipynb

PINN simulation template for **free-free** 20-DOF chain with **no external force** and **initial velocity at DOF 1**.


In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from pinn_ndof_chain_tf2_free_free_no_force import (
    PIPNNs,
    build_free_free_chain_matrices,
    make_left_velocity_ic,
    find_impact_times,
    impact_velocity_update,
    propagate_ics,
    newmark_beta,
)

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
np.random.seed(1234)
tf.random.set_seed(1234)
print('TF version:', tf.__version__)


## Parameters (free-free, no external force)


In [ ]:
n_dof = 20
m_x = 1.0
m_y = 0.3
k = 1.0
c = 0.0
D = 1.0
r = 1.0

# no external forcing in this variant
phi = 0.0
phi1 = 0.0
phi2 = 0.0

# left-end initial condition (wave launch)
x1_0 = 0.0
v1_0 = 1.0

M_total, C_total, K_total = build_free_free_chain_matrices(
    n_dof=n_dof, m_x=m_x, k=k, c=c
)

x0_total, xt0_total = make_left_velocity_ic(
    n_dof=n_dof, x1_0=x1_0, v1_0=v1_0
)

# impactor initial conditions
y0 = np.zeros((1, n_dof))
yt0 = np.zeros((1, n_dof))

print('x0_total shape:', x0_total.shape)
print('xt0_total shape:', xt0_total.shape)
print('left dof IC:', x0_total[0,0], xt0_total[0,0])


## Newmark reference (no external force)
Use this to quickly verify wave propagation from left to right.


In [ ]:
T_ref = 8.0
dt_ref = 0.001
num_ref = int(T_ref / dt_ref) + 1
t_ref = np.linspace(0.0, T_ref, num_ref).reshape(-1, 1)

F_ref = np.zeros((n_dof, num_ref))  # strictly no external force

u_NM_ref, ut_NM_ref, _ = newmark_beta(
    M_total, C_total, K_total, F_ref, dt_ref, num_ref, n_dof,
    x0=x0_total.flatten(), xt0=xt0_total.flatten()
)

plt.figure(figsize=(9, 3.5))
plt.plot(t_ref, u_NM_ref[0, :], label='DOF1 (left)')
plt.plot(t_ref, u_NM_ref[-1, :], label='DOF20 (right)')
plt.xlabel('Time (s)')
plt.ylabel('Displacement')
plt.title('Free-free wave propagation check (Newmark, no force)')
plt.legend(); plt.grid(alpha=0.25); plt.tight_layout(); plt.show()


## PINN segment setup
This mirrors your existing PINN workflow, but with free-free matrices and zero forcing.


In [ ]:
# Segment time domain
lb = np.array([0.0])
ub = np.array([1.0])

N_f = 2000
N0 = 1

t = np.random.rand(N_f, 1) * (ub - lb) + lb
t0 = np.zeros((N0, 1))

layers = [1, 64, 64, n_dof]
hyp_ini_weight_loss = [1.0, 1.0]

model = PIPNNs(
    lb=lb,
    ub=ub,
    t0=t0,
    t=t,
    x0_total=x0_total,
    xt0_total=xt0_total,
    y0=y0,
    yt0=yt0,
    M=M_total,
    K=K_total,
    C=C_total,
    D=D,
    n_dof=n_dof,
    phi=phi,
    phi1=phi1,
    phi2=phi2,
    layers=layers,
    hyp_ini_weight_loss=hyp_ini_weight_loss,
    optimizer_LB=True,
)

print('Model built (free-free, no force).')


## Train and impact detection (optional)
Uncomment and run according to your compute budget.


In [ ]:
# Example (optional):
# model.train(epochs=20000)
# t_impact, x_impact, xt_impact, y_impact, yt_impact = find_impact_times(model)
# print('First impact times:', t_impact)
